# Postprocessing tutorial

Welcome to this tutorial on Roman CGI postprocessing. In this guide, we'll walk through the combination of roll (ADI) and chop (RDI) images, previously simulated with corgisim.
The processing is done in a very simple manner as a sake of understanding. Feel free to use your favorite tool at any time to go further.

In [ ]:
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.io.fits as fits

# Load data image
We previously saved data that were generated with corgisim (two rolls and one ref star acquisition)
Here we coadd the data obtained at same epoch.
Reminder that the unit is photons/sec
The images do not correspond to raw data obtained with CGI. We assume they are preprocessed (cropped around the star located at the central pixel).

In [ ]:
# Start with defining a function that extract the images and create a coadd image in photon/sec
def coadd_epoch(directory, name_start_with):
    # Get all the files in directory whose filename start with name_start_with
    files = [f for f in os.listdir(directory) if f.startswith(name_start_with)]

    # Extract data and sort in cube
    data_cube = []
    for file in files:
        data_cube.append(fits.getdata(file))
    data_cube = np.array(data_cube)

    #Coadd data
    coadd_data = np.mean(data_cube, axis = 0)
    return coadd_data


# Extracting the two rolls data sets
data_dir = os.getcwd()
coadd_roll1 = coadd_epoch(data_dir, 'host_star_image_roll1')
coadd_roll2 = coadd_epoch(data_dir, 'host_star_image_roll2')

plt.figure(figsize=(16,6))

plt.subplot(121)
plt.imshow(coadd_roll1)
plt.title('Epoch 1, roll 1 (0 deg)')
plt.colorbar()

plt.subplot(122)
plt.title('Epoch 1, roll 2 (30 deg)')
plt.imshow(coadd_roll2)
plt.colorbar()

Back in business..

# Roll subtraction and roll combination
Taking the direct differences between the co-adds acquired in opposite roll states reveals a preliminary point source detection, and confirms the relative orientation of the roll.

Relative to roll 1, the sky in roll 2 appears rotated 30 degrees counter-clockwise.

In [ ]:
diff1 = coadd_roll1 - coadd_roll2
diff2 = coadd_roll2 - coadd_roll1

plt.figure(figsize=(16,6))

plt.subplot(121)
plt.imshow(diff1)
plt.title('Epoch 1, roll 1 - roll 2')
plt.colorbar()

plt.subplot(122)
plt.title('Epoch 1, roll 2 - roll 1')
plt.imshow(diff2)
plt.colorbar()

## Derotate the second difference and sum with the first difference

In [ ]:
import skimage.transform
scistar_coadd_ep1_rolls = [0, 30]
imgwidth = diff2.shape[0]
derot_diff_2 = skimage.transform.rotate(image = diff2,
                                        angle = -scistar_coadd_ep1_rolls[1],
                                        center = (imgwidth // 2, imgwidth // 2),
                                        order = 3, cval = np.nan)
derot_sum = diff1 + derot_diff_2

plt.figure(figsize=(8,6))

plt.imshow(derot_sum)

plt.title('Epoch 1, (roll 1 - roll 2) + derot(roll 2 - roll 1)')
plt.colorbar()

## Question 1: How would you estimate the signal-to-noise ratio of this source?

# Reference Differential Imaging

Plot the reference star along with the host star (roll1)

In [ ]:
coadd_ref = coadd_epoch(data_dir, "ref_star_image")

plt.figure(figsize=(16,6))

plt.subplot(121)
plt.imshow(coadd_roll1)
plt.title('Epoch 1, roll 1 (0 deg)')
plt.colorbar()

plt.subplot(122)
plt.title('Epoch 1, Reference image')
plt.imshow(coadd_ref)
plt.colorbar()


## Compute reference-to-science count ratio (crude but good enough for a first-pass inspection)

In [ ]:
count_sum_ratio_roll1 = (np.sum(coadd_ref)
                    / np.sum(coadd_roll1))
count_sum_ratio_roll2 = (np.sum(coadd_ref)
                    / np.sum(coadd_roll2))
print(count_sum_ratio_roll1)
print(count_sum_ratio_roll2)

In [ ]:
crude_rdi_result_roll1 = (coadd_roll1
                    - coadd_ref / count_sum_ratio_roll1)
crude_rdi_result_roll2 = (coadd_roll2
                    - coadd_ref / count_sum_ratio_roll2)

plt.figure(figsize=(16,6))

plt.subplot(121)
plt.imshow(crude_rdi_result_roll1)
plt.title('Epoch 1, roll 1 - ref')
plt.colorbar()

plt.subplot(122)
plt.title('Epoch 1, roll 2 - ref')
plt.imshow(crude_rdi_result_roll2)
plt.colorbar()

## Question 2: what can be done to improve this subtraction

# Quick look photometry & flux ration calibration demo
Estimate the flux ratio of a point source in an HLC image based on aperture photometry in a 3x3 pixel box

In [ ]:
# Pick one result image to analyze
processed_image = crude_rdi_result_roll1
img_width = processed_image.shape[0]

#Pixel box
phot_box_width = 3

#Find maximum index in processed image
src_peak_row = np.nanargmax(np.ravel(processed_image)) // img_width 
src_peak_col = np.nanargmax(np.ravel(processed_image)) % img_width

#Compute nb of counts
phot_box_sum = np.sum(processed_image[src_peak_row - phot_box_width // 2 : 
                                    src_peak_row + phot_box_width // 2 + 1,
                                    src_peak_col - phot_box_width // 2 : 
                                    src_peak_col + phot_box_width // 2 + 1]) * u.photon/u.second

print("Peak col = {:d}, peak row = {:d}".format(src_peak_col, src_peak_row))
print("Sum in {:d} x {:d} box = {:.5f}".format(phot_box_width, phot_box_width, phot_box_sum))

plt.figure(figsize=(12, 4))

plt.subplot(121)
plot_box_width = 7
plt.imshow(processed_image[src_peak_row - plot_box_width // 2 : 
                         src_peak_row + plot_box_width // 2 + 1,
                         src_peak_col - plot_box_width // 2 : 
                         src_peak_col + plot_box_width // 2 + 1])
_ = plt.colorbar()

plt.subplot(122)
plt.imshow(processed_image[src_peak_row - phot_box_width // 2 : 
                         src_peak_row + phot_box_width // 2 + 1,
                         src_peak_col - phot_box_width // 2 : 
                         src_peak_col + phot_box_width // 2 + 1])
_ = plt.colorbar()

## Estimate the background count rate by taking a box away from the source, but at a similar angular separation from the star

In [ ]:
bg_samp_col = 32
bg_samp_row = 22
bg_box_width = 5
bg_est = np.median(processed_image[bg_samp_row - bg_box_width//2 : 
                                 bg_samp_row + bg_box_width // 2 + 1,
                                 bg_samp_col - bg_box_width//2 : 
                                 bg_samp_col + bg_box_width // 2 + 1]) * u.photon/u.second

phot_box_sum_bgsub = phot_box_sum - bg_est #* (phot_box_width * phot_box_width)

print("B.g. estimate = {:.5f}".format(bg_est))
print("B.g.-subtracted aperture sum = {:.5f}".format(phot_box_sum_bgsub))

plt.figure(figsize=(12, 4))

plt.subplot(121)
plot_box_width = 10
plt.imshow(processed_image[bg_samp_row - plot_box_width//2 : 
                         bg_samp_row + plot_box_width // 2 + 1,
                         bg_samp_col - plot_box_width//2 : 
                         bg_samp_col + plot_box_width // 2 + 1])
_ = plt.colorbar()

plt.subplot(122)
plt.imshow(processed_image[bg_samp_row - bg_box_width//2 : 
                         bg_samp_row + bg_box_width // 2 + 1,
                         bg_samp_col - bg_box_width//2 : 
                         bg_samp_col + bg_box_width // 2 + 1])
_ = plt.colorbar()

## Measure the star PSF model count rate in the same 3x3 box

In [ ]:
# Extract and coadd PSF data
host_star_PSF = coadd_epoch(data_dir, "host_star_PSF")

host_star_PSF_width = host_star_PSF.shape[0]

star_peak_row = np.nanargmax(np.ravel(host_star_PSF)) // host_star_PSF_width
star_peak_col = np.nanargmax(np.ravel(host_star_PSF)) % host_star_PSF_width
print(star_peak_col, star_peak_row)

star_phot_box_sum = (np.sum(host_star_PSF[star_peak_row - phot_box_width//2 : 
                                         star_peak_row + phot_box_width // 2 + 1,
                                         star_peak_col - phot_box_width//2 : 
                                         star_peak_col + phot_box_width // 2 + 1])
                     * u.photon / u.second)

print("Peak col = {:d}, peak row = {:d}".format(star_peak_col, star_peak_row))
print("Star PSF photon count rate in {:d} x {:d} box = {:.2f}".format(
      phot_box_width, phot_box_width, star_phot_box_sum))

plt.figure(figsize=(8, 6))
plt.imshow(host_star_PSF)
plt.colorbar()

## Estimate the relative PSF attenuation due to the occulting mask

In [ ]:
# Extract a built-in psf peak throughput map (only HLC)
psf_peak_map = fits.getdata(data_dir + '/OS6_HLC_PSF_peak_map.fits')
psf_peak_map_hdr = fits.getheader(data_dir + '/OS6_HLC_PSF_peak_map.fits')

# Translate pixel of the detector to pixel of PSF peak map array
pix_size_corgisim = 21.8/49.3 #In (λ/D)/pixel.   Assumptions: 21.8 mas/pix on excam /// 49.3 mas/(λ/D) at 550nm
pixscale_ratio =  pix_size_corgisim / psf_peak_map_hdr['PIXSIZE']

peak_map_width = psf_peak_map.shape[0]

src_peak_map_col = (peak_map_width // 2 
                    + int(np.round((src_peak_col - img_width // 2)
                    * pixscale_ratio)))
src_peak_map_row = (peak_map_width // 2
                    + int(np.round((src_peak_row - img_width // 2)
                    * pixscale_ratio)))

print("Source position in peak map: {:}, {:}".format(src_peak_map_col, src_peak_map_row))

#Retrieve HLC throughput, assuming maximum throughput=1 
psf_atten = psf_peak_map[src_peak_map_row, src_peak_map_col] / np.max(psf_peak_map)
print("Relative PSF attenuation: {:.2f}".format(psf_atten))

## Compare measured planet flux ration against truth

In [ ]:
planet_phot_box_sum = phot_box_sum_bgsub #/ 1)/ u.electron * u.photon / u.second#(det_qe * pc_loss * psf_atten)) / tot_inttime
print("Planet PSF photon count rate in {:d} x {:d} box = {:.2E}".format(
      phot_box_width, phot_box_width, planet_phot_box_sum))

planet_flux_ratio = planet_phot_box_sum/star_phot_box_sum/psf_atten #(planet_phot_box_sum / 10000) / (star_phot_box_sum / 0.01)  /0.9
print("Measured planet flux ratio = {:.2E}".format(planet_flux_ratio))

true_flux_ratio = 10**((7-25)/2.5)  # taken from scene simulation inputs
error_frac = (planet_flux_ratio - true_flux_ratio) / true_flux_ratio
print("Measured flux ratio = {:.2E}\nTrue flux ratio = {:.2E}\nRelative error = {:.1f}%".format(
      planet_flux_ratio, true_flux_ratio, 100 * error_frac))

# Processing with VIP
To go a bit further, and waiting for the official corgi data reduction pipeline, we can also test advanced postprocessing with our favorite tool (VIP, PyKLIP, Pynpoint, ...). In this section, VIP is used and many things can be explored.
To install VIP in your environment, simply write `pip install vip_hci`
Tutorial can be found at https://vip.readthedocs.io/en/latest/index.html

In [ ]:
import numpy as np
import vip_hci
from os import mkdir
from os.path import join, sep,exists
import glob
import matplotlib

%matplotlib inline

from vip_hci.var import fit_2dgaussian
from hciplot import plot_frames, plot_cubes
from vip_hci.preproc import frame_crop
from vip_hci.fits import open_fits, write_fits

from vip_hci.psfsub import pca
from vip_hci.preproc import cube_crop_frames, frame_crop, cube_derotate
from vip_hci.fits import write_fits

In [ ]:
# Full frame PCA
inner_rad_adi = 3 #lam/D

roll_images = np.array([coadd_roll1, coadd_roll2])
roll_angles = np.array(scistar_coadd_ep1_rolls)


pca1 = pca(roll_images, roll_angles, cube_ref=None, ncomp=1, 
           svd_mode='lapack', mask_center_px=inner_rad_adi, full_output=False)

plot_frames(pca1, grid=True)#, backend='bokeh')

## Now let's look at the recorded PSF

In [ ]:
from vip_hci.fm import normalize_psf  # fm is the forward modeling subpackage, with various tools for the characterization of point sources and extended signals.

#Measures the flux anf fwhm of the PSF
psfn, flux, fwhm = normalize_psf(host_star_PSF, size=19, debug=True, full_output=True)

In [ ]:
#VIP package unables to automatically find significant point sources
detect = vip_hci.metrics.detection(pca1, psf=psfn, debug=False,mode='log', snr_thresh=3, 
                      bkg_sigma=4, matched_filter=False)

In [ ]:
from vip_hci.metrics import snr
xy_b = (18.14,18.24) # Set the position of the discovered point source
snr1 = snr(pca1, source_xy=xy_b, fwhm=fwhm) # Provides SNR detection
print(r"S/N = {:.1f}".format(snr1))

In [ ]:
#We can also obtain a SNR map for the all field of view
from vip_hci.metrics import snrmap

snrmap1 = snrmap(pca1, fwhm, plot=True)

### Did you discover all your injected planets in your images?
## Photometry with VIP - Forward modeling
Another way to compute the photometry of the discovered planets is to inject fake planets in the data

In [ ]:
## Make fake planets
from vip_hci.fm import cube_inject_companions 

imlib_rot = 'skimage'      # If you have opencv installed, feel free to set this parameter to" 'opencv'
interpolation= 'biquintic'   # If you have opencv installed, feel free to set this parameter to 'lanczos4'

pxscale=0.0210804
blank=np.zeros(((2,45,45)))

#This function injects a fake companion in the images. Here a flux level of 5e-3 photon/sec
cubefc = cube_inject_companions(roll_images, psfn, roll_angles, flevel=5e-3, plsc=pxscale, 
                                rad_dists=[15], theta=180, n_branches=1)
pca1_fc = pca(cubefc, roll_angles, ncomp=1)

plot_frames(pca1_fc)
                                 

Such injection can also be performed negatively, at the detected source position. The flux of the detected source corresponds to the minimum residual obtained after subtraction.
With the function firstguess, we can obtain a first estimation of the flux and position by running A) a naive grid minimization (grid of values for the flux through parameter f_range), and B) a Nelder-mead based minimization (if the parameter simplex is set to True). The latter is done based on the preliminary guess of the grid minimization. 

In [ ]:
from vip_hci.fm import firstguess

source_xy = (18.14,18.24)
ncomp_fc = 1
factor = 1000 #to help flux computation

r_0, theta_0, f_0 = firstguess(roll_images*factor, roll_angles, psfn, ncomp=ncomp_fc, planets_xy_coord=[source_xy], 
                                  fwhm=fwhm, f_range=None, annulus_width=4*fwhm, 
                                  aperture_radius=2, simplex=True, imlib=imlib_rot, 
                                  interpolation=interpolation, plot=True, verbose=True)
f_0 = f_0/factor
print('Radial distance = ', r_0, ' l/D')
print('Angle = ',  theta_0, ' degree')
print('Flux = ', f_0, ' photon/sec')

In [ ]:
#Let's measure the obtained flux ratio, still taking the HLC attenuation at source position into account
planet_flux_ratio_vip = f_0[0]/flux[0]/psf_atten #(planet_phot_box_sum / 10000) / (star_phot_box_sum / 0.01)  /0.9
print("Measured planet flux ratio = {:.2E}".format(planet_flux_ratio_vip))

true_flux_ratio = 10**((7-25)/2.5)  # taken from scene simulation inputs
error_frac = (planet_flux_ratio_vip - true_flux_ratio) / true_flux_ratio
print("Measured flux ratio = {:.2E}\nTrue flux ratio = {:.2E}\nRelative error = {:.1f}%".format(
      planet_flux_ratio_vip, true_flux_ratio, 100 * error_frac))

## A quick MCMC to end the day
Markov Chain Monte Carlo (MCMC) is a robust way of obtaining the flux and position of the companion, and their uncertainties. MCMC samples the parameter space (here r and  coordinates, and flux) using a number of walkers following a Markov chain (i.e. a process where each new sampled point only depends on the previous one) for a certain number of iterations. At each iteration, a new sample of parameters is proposed to each walker to explore the parameter space
Beware that the MCMC procedure is a CPU intensive procedure and can take several hours when run properly on a standard laptop. 


In [ ]:
from vip_hci.psfsub import pca_annulus

#factor to help the mcmc to converge
factor = 1e5
#Let’s first define observation-related parameters, such as the non-coronagraphic psf, its FWHM and the pixel scale of the detector:
obs_params = {'psfn': psfn,
              'fwhm': fwhm}

#In NEGFC, PCA in a single annulus is used by default to speed up the algorithm - although other algorithms can be used through the algo parameter.
#Let’s set the  to the optimal  found in Section 5.2. We set the width of the annulus on which PCA is performed (in pixels) with the annulus_width parameter.
# We also set a few other algorithm-related parameters in the following:
annulus_width = 4*fwhm

algo_params = {'algo': pca_annulus,
               'ncomp': 1,
               'annulus_width': annulus_width,
               'svd_mode': 'lapack',
               'imlib': imlib_rot, 
               'interpolation': interpolation}

mu_sigma=True
aperture_radius=2

negfc_params = {'mu_sigma': mu_sigma,
                'aperture_radius': aperture_radius}

initial_state = np.array([r_0[0], theta_0[0], f_0[0]*factor])
print(initial_state)

from multiprocessing import cpu_count

nwalkers, itermin, itermax = (100, 200, 500)

mcmc_params = {'nwalkers': nwalkers,
               'niteration_min': itermin,
               'niteration_limit': itermax,
               'bounds': None,
               'nproc': cpu_count()//2}

conv_test, ac_c, ac_count_thr, check_maxgap = ('ac', 50, 1, 50)

conv_params = {'conv_test': conv_test,
               'ac_c': ac_c,
               'ac_count_thr': ac_count_thr,
               'check_maxgap': check_maxgap}

from vip_hci.fm import mcmc_negfc_sampling
chain = mcmc_negfc_sampling(roll_images*factor, roll_angles, **obs_params, **algo_params, **negfc_params, 
                            initial_state=initial_state, **mcmc_params, **conv_params,
                            display=True, verbosity=2, save=False, output_dir='./')

In [ ]:
from vip_hci.fm import show_corner_plot
burnin = 0.3
burned_chain = chain[:, int(chain.shape[1]//(1/burnin)):, :]
show_corner_plot(burned_chain)

In [ ]:
#Now let’s determine the most highly probable value for each model parameter, as well as the 1-sigma confidence interval. For this, let’s first flatten the chains:
isamples_flat = chain[:, int(chain.shape[1]//(1/burnin)):, :].reshape((-1,3))
from vip_hci.fm import confidence
val_max, conf = confidence(isamples_flat, cfd=68, gaussian_fit=False, verbose=True, save=False, 
                         ndig=1, title=True)

Again, the resulting flux can be compared with the ground truth in simulations
# This tutotial is now completed